In [1]:
from pathlib import Path
import json
import pandas as pd

# Display settings (optional)
pd.set_option('display.max_colwidth', 120)

# How many rows to show when displaying DataFrames in the notebook
pd.set_option('display.max_rows', 200)  # increase if you want more
pd.set_option('display.min_rows', 50)

# Resolve repository root from this notebook location.
repo_root = Path.cwd()
if repo_root.name == "PyDataTransform":
    repo_root = repo_root.parent

json_path = repo_root / "JSON Whole Model" / "ASTIDC-STAN-HE-MPD-TXBP1-M-M-0001.json"
assert json_path.exists(), f"JSON file not found: {json_path}"

with json_path.open("r", encoding="utf-8") as f:
    whole_model_data = json.load(f)

# Quick confirmation output
print(f"Loaded JSON: {json_path}")
if isinstance(whole_model_data, list):
    print(f"Top-level type: list, items: {len(whole_model_data)}")
elif isinstance(whole_model_data, dict):
    print(f"Top-level type: dict, keys: {len(whole_model_data)}")
    print("First keys:", list(whole_model_data.keys())[:10])
else:
    print(f"Top-level type: {type(whole_model_data).__name__}")

Loaded JSON: c:\Git\APS-IFC\JSON Whole Model\ASTIDC-STAN-HE-MPD-TXBP1-M-M-0001.json
Top-level type: list, items: 6580


In [2]:
from pathlib import Path
import json
import shutil
import pandas as pd

# Match rename-notebook workflow: copy source JSON into JSON_Edit first.
file_name = "ASTIDC-STAN-HE-MPD-TXBP1-M-M-0001.json"
source_path = repo_root / "JSON Whole Model" / file_name
assert source_path.exists(), f"Source file not found: {source_path}"

json_edit_dir = repo_root / "JSON_Edit"
json_edit_dir.mkdir(parents=True, exist_ok=True)

working_json_path = json_edit_dir / file_name
shutil.copy2(source_path, working_json_path)
print(f"Copied source JSON to: {working_json_path}")

with working_json_path.open("r", encoding="utf-8") as f:
    records = json.load(f)

df = pd.DataFrame(records)
print(f"Working DataFrame shape: {df.shape}")

Copied source JSON to: c:\Git\APS-IFC\JSON_Edit\ASTIDC-STAN-HE-MPD-TXBP1-M-M-0001.json
Working DataFrame shape: (6580, 4)


#### Model element Name counts
This notebook loads a `JSON Whole Model/*.json` export and prints a table with **Name**, **DbId**, **GUID**, and the **total count of that Name** across the model (duplicates included).

In [7]:
def _extract_guid(props):
    if not isinstance(props, list):
        return None
    for item in props:
        if not isinstance(item, dict):
            continue
        display_name = str(item.get('displayName', '')).strip()
        if display_name.lower() == 'guid':
            return item.get('value')
    return None

# Build GUID + name counts table
df['GUID'] = df['Properties'].apply(_extract_guid)
name_counts = df['Name'].value_counts(dropna=False)
df['NameCount'] = df['Name'].map(name_counts)

table = (
    df[['Name', 'DbId', 'GUID', 'NameCount']]
    .sort_values(['Name', 'DbId'], kind='stable')
    .reset_index(drop=True)
)

rows_to_show = 50  # set to None to display all rows
print(f"Total rows in table: {len(table)}")
table if rows_to_show is None else table.head(rows_to_show)

Total rows in table: 6580


,Name,DbId,GUID,NameCount
0,#4662766,1013,None,1
1,#4662804,1689,None,1
2,1JNL100332-220_00-Rubber Plate,5014,225154f9-31fd-32e5-b35f-82d1d1a7165a,288
3,1JNL100332-220_00-Rubber Plate,5015,651ab0ea-4446-3764-b7b9-657736c28330,288
4,1JNL100332-220_00-Rubber Plate,5016,74c01236-6f37-32c9-a625-306be1debbb3,288
5,1JNL100332-220_00-Rubber Plate,5017,8d88dfc1-242d-3f6f-839c-dc416b312455,288
6,1JNL100332-220_00-Rubber Plate,5034,edc2be86-7af8-3602-8c45-b2a785e8bcc6,288
7,1JNL100332-220_00-Rubber Plate,5035,5335ee00-1166-39b0-a6f6-7a89354a41e1,288
8,1JNL100332-220_00-Rubber Plate,5036,74542c6e-6eb0-3454-8421-3adb1cb79c53,288
9,1JNL100332-220_00-Rubber Plate,5037,2dc03adc-5b58-31c5-8d4a-814d14f9c55b,288
